# AuraGateway current CUDA 12.9 harness materializer

Attach exactly one reviewed source input. Exact ZIP and Kaggle auto-expanded source representations are supported. Use Accelerator None, Internet Off, no secrets, and Save Version -> Save & Run All.


In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import stat
import zipfile
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = 'ag-harness-materializer-cu129-v1'
DATASET_NAME = 'ag-harness-4f3302d-v1-input'
INPUT_ROOT = Path("/kaggle/input").resolve()
WORK_ROOT = Path("/kaggle/working").resolve()
EXPECTED_ARCHIVE_NAME = 'ag-harness-4f3302d-v1.zip'
EXPECTED_EXPANDED_DIRECTORY = EXPECTED_ARCHIVE_NAME.removesuffix(".zip")
EXPECTED_SOURCE_COMMIT = '4f3302df871d47fec81e25e9af9609c0e2c7812d'
EXPECTED_DIRECTORY_SHA256 = 'a154f3453c55571fc7535b546e4a97a66756ceb1900b51c2fd1336fed981d307'
EXPECTED_FILE_COUNT = 1095
EXPECTED_TOTAL_BYTES = 11034996
EXPECTED_OUTPUT_DIRECTORY = 'auragateway_qualification_harness_4f3302d_v1'
EXPECTED_MATERIALIZATION_RECEIPT_NAME = 'ag_harness_materialization_receipt_cu129_v1.json'
EXPECTED_CONTROL_FILES = (
    "source_inventory.json",
    "source_packaging_receipt.json",
    "sha256_manifest.json",
)
EXPECTED_CONTROL_SHA256 = {'ag-harness-4f3302d-v1.zip': 'a0872f6e7b4b9c60522fdb55d2248b0143d27c6553a4e2aff423d377c0da09b1', 'source_inventory.json': '56204e798d859a013c740d6fca4728316775568af9a0fdc0c4a02c426e61eb0d', 'source_packaging_receipt.json': 'cf1d472e5fc2e141c3369d85578350dcb80af8e095f42440c1993f3f22b56648', 'sha256_manifest.json': '1c850e21b896a8e151e8ddcc52ba10b86e2f6f0f36979181879e95747729f4f6'}
EXPECTED_SOURCE_RECEIPT = {'schema_version': '1.0.0', 'status': 'CURRENT_CU129_HARNESS_SOURCE_PACKAGED', 'package_id': 'auragateway-cu129-current-harness-toolchain-v1', 'source_commit': '4f3302df871d47fec81e25e9af9609c0e2c7812d', 'archive_name': 'ag-harness-4f3302d-v1.zip', 'archive_sha256': 'a0872f6e7b4b9c60522fdb55d2248b0143d27c6553a4e2aff423d377c0da09b1', 'inventory_name': 'source_inventory.json', 'inventory_sha256': '56204e798d859a013c740d6fca4728316775568af9a0fdc0c4a02c426e61eb0d', 'source_receipt_name': 'source_packaging_receipt.json', 'sha256_manifest_name': 'sha256_manifest.json', 'output_directory': 'auragateway_qualification_harness_4f3302d_v1', 'input_dataset_name': 'ag-harness-4f3302d-v1-input', 'materialization_receipt_name': 'ag_harness_materialization_receipt_cu129_v1.json', 'directory_sha256': 'a154f3453c55571fc7535b546e4a97a66756ceb1900b51c2fd1336fed981d307', 'file_count': 1095, 'total_bytes': 11034996, 'required_paths': ['pyproject.toml', 'ruff.toml', 'README.md', 'src/auragateway/local_abc/contracts.py', 'src/auragateway/local_abc/errors.py', 'src/auragateway/local_abc/full_abc_local_environment_qualification_execution.py', 'src/auragateway/local_abc/full_abc_local_environment_qualification_execution_contracts.py', 'src/auragateway/local_abc/full_abc_local_environment_qualification_execution_authorization_issuance.py', 'src/auragateway/local_abc/full_abc_local_environment_qualification_kaggle_launcher.py', 'src/auragateway/local_abc/full_abc_local_environment_qualification_kaggle_runtime_adapter.py', 'src/auragateway/local_abc/full_abc_local_environment_qualification_cu129_runtime.py', 'src/auragateway/local_abc/full_abc_local_environment_qualification_cu129_harness_rematerialization_review.py', 'data/evals/benchmark/environment-qualification-v1/offline_dataset_manifest.json', 'data/evals/benchmark/environment-qualification-v1/offline_dataset_materialization_record.json', 'data/evals/benchmark/environment-qualification-v1/qualification_execution_request.json', 'data/evals/benchmark/environment-qualification-v1/worker_startup_plan.json', 'notebooks/auragateway_full_abc_environment_qualification_v1.ipynb', 'notebooks/auragateway_full_abc_environment_qualification_launcher_v1.ipynb', 'benchmarks/local_abc/auragateway_cu129_current_harness_rematerialization_review_v1.json', 'benchmarks/local_abc/auragateway_cu129_current_harness_toolchain_v1.json', 'docs/reports/AuraGateway_CU129_Harness_Toolchain_Shared_Authority_Propagation_Reasoning_Certificate.md', 'src/auragateway/local_abc/full_abc_local_environment_qualification_cu129_harness_toolchain.py', 'src/auragateway/local_abc/full_abc_local_environment_qualification_cu129_harness_toolchain_contracts.py', 'src/auragateway/local_abc/full_abc_local_environment_qualification_cu129_vllm_cli_contract_hardening.py', 'benchmarks/local_abc/auragateway_cu129_vllm_cli_contract_hardening_v1.json', 'docs/adr/2026-07-27-local-abc-cu129-vllm-cli-contract-hardening.md', 'docs/reports/AuraGateway_CU129_VLLM_CLI_Contract_Hardening_Report.md', 'docs/runbooks/local_abc_cu129_vllm_cli_contract_hardening_v1.md'], 'expected_file_sha256': {'src/auragateway/local_abc/full_abc_local_environment_qualification_execution.py': '0851a3819806af89b4e6ae86faa8bfb6949db46c4436ebc2a580be92f0a0950b', 'src/auragateway/local_abc/full_abc_local_environment_qualification_kaggle_launcher.py': 'b913c8c24bda8b5a6478a9f2b6720cc0e30abc2344352d4bc6e66360c57493db', 'notebooks/auragateway_full_abc_environment_qualification_launcher_v1.ipynb': '138b5a04185082aeb671f1be5511ebdf4e4da00970eaa6145fbbd953d567c44c', 'src/auragateway/local_abc/full_abc_local_environment_qualification_kaggle_runtime_adapter.py': 'f83452b6fbfd583f4236c2edbaf0e4bd3a6ece331494fdff891bf50d022ba617', 'src/auragateway/local_abc/full_abc_local_environment_qualification_cu129_runtime.py': '5cf0379c514a023a100230ff44be3297c6381740ede44bb27a4288adbb2c174f', 'src/auragateway/local_abc/full_abc_local_environment_qualification_execution_contracts.py': 'e88fead2fc576dd2965a666affbe8e3669b928ffc1a51b9baae178e02172af02', 'data/evals/benchmark/environment-qualification-v1/qualification_execution_request.json': '3079f1577588fa10ae47689a06bec96615b94ba5b0e42ee6bb0d2fed625357a8', 'data/evals/benchmark/environment-qualification-v1/worker_startup_plan.json': '3a987c755e9be1a480a6f70fc667d2813eecc183924ba956fd12cae45d5eb8f8', 'notebooks/auragateway_full_abc_environment_qualification_v1.ipynb': '5d166eddaff724d0daa5ed69c69b581e8c0403c212d0605f3f75e7294389240e'}, 'safety': {'network_access_performed': False, 'package_installation_performed': False, 'gpu_execution_performed': False, 'model_loaded': False, 'tokenizer_loaded': False, 'worker_started': False, 'model_requests_performed': 0, 'benchmark_trajectory_requests_performed': 0, 'authorization_issued': False, 'credentials_present': False, 'customer_data_present': False, 'external_spend': 0}}
EXPECTED_SHA256_MANIFEST = {'ag-harness-4f3302d-v1.zip': 'a0872f6e7b4b9c60522fdb55d2248b0143d27c6553a4e2aff423d377c0da09b1', 'source_inventory.json': '56204e798d859a013c740d6fca4728316775568af9a0fdc0c4a02c426e61eb0d', 'source_packaging_receipt.json': 'cf1d472e5fc2e141c3369d85578350dcb80af8e095f42440c1993f3f22b56648'}
PRODUCER_OUTPUT_DIRECTORY = "ag_harness_materializer_cu129_v1_output"
FINAL_BUNDLE_ROOT = WORK_ROOT / PRODUCER_OUTPUT_DIRECTORY
STAGING_BUNDLE_ROOT = WORK_ROOT / ".ag_harness_materializer_cu129_v1_staging"
STAGING_HARNESS_ROOT = STAGING_BUNDLE_ROOT / EXPECTED_OUTPUT_DIRECTORY
STAGING_RECEIPT_PATH = (
    STAGING_BUNDLE_ROOT / EXPECTED_MATERIALIZATION_RECEIPT_NAME
)
RECOVERED_ARCHIVE_PATH = (
    WORK_ROOT / ".ag_harness_materializer_cu129_v1_recovered.zip"
)
MAXIMUM_FILES = 5000
MAXIMUM_TOTAL_BYTES = 104857600
ARCHIVE_SUFFIXES = ('.zip', '.tar', '.tgz', '.gz', '.bz2', '.xz', '.7z', '.whl')
ZIP_TIMESTAMP = (1980, 1, 1, 0, 0, 0)


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def normalized_relative_path(value: str) -> PurePosixPath:
    path = PurePosixPath(value)
    if (
        path.is_absolute()
        or not path.parts
        or ".." in path.parts
        or "\\" in value
        or value.startswith("./")
        or path.as_posix() != value
    ):
        raise RuntimeError("source dataset contains an unsafe relative path")
    return path


def resolve_source_package() -> tuple[Path, str, Path]:
    candidates: list[tuple[Path, str, Path]] = []
    for receipt_path in INPUT_ROOT.rglob("source_packaging_receipt.json"):
        if not receipt_path.is_file() or receipt_path.is_symlink():
            continue
        dataset_root = receipt_path.parent.resolve()
        if INPUT_ROOT not in dataset_root.parents:
            continue
        controls_present = all(
            (dataset_root / name).is_file()
            and not (dataset_root / name).is_symlink()
            for name in EXPECTED_CONTROL_FILES
        )
        if not controls_present:
            continue

        archive_path = dataset_root / EXPECTED_ARCHIVE_NAME
        expanded_root = dataset_root / EXPECTED_EXPANDED_DIRECTORY
        representations = []
        if archive_path.is_file() and not archive_path.is_symlink():
            representations.append(("exact_archive_with_control_files", archive_path))
        if expanded_root.is_dir() and not expanded_root.is_symlink():
            representations.append(
                (
                    "kaggle_expanded_source_recovered_to_exact_archive",
                    expanded_root,
                )
            )
        if len(representations) != 1:
            continue
        input_mode, representation_path = representations[0]
        candidates.append(
            (dataset_root, input_mode, representation_path.resolve())
        )

    unique = tuple(dict.fromkeys(candidates))
    if len(unique) != 1:
        observed = tuple(
            str(path.relative_to(INPUT_ROOT))
            for path in sorted(
                INPUT_ROOT.rglob("source_packaging_receipt.json"),
                key=lambda item: item.as_posix(),
            )
            if path.is_file()
        )
        raise RuntimeError(
            "expected exactly one identity-shaped harness source package "
            f"but observed {len(unique)}; receipt_candidates={observed!r}"
        )
    return unique[0]


def validate_dataset_controls(
    dataset_root: Path,
    input_mode: str,
) -> list[dict[str, object]]:
    representation_name = (
        EXPECTED_ARCHIVE_NAME
        if input_mode == "exact_archive_with_control_files"
        else EXPECTED_EXPANDED_DIRECTORY
    )
    expected_names = set(EXPECTED_CONTROL_FILES) | {representation_name}
    observed_names = set()
    for path in dataset_root.iterdir():
        if path.is_symlink():
            raise RuntimeError("source dataset contains a symbolic link")
        metadata = path.stat()
        if not stat.S_ISREG(metadata.st_mode) and not path.is_dir():
            raise RuntimeError(
                "source dataset contains a non-regular top-level member"
            )
        observed_names.add(path.name)
    if observed_names != expected_names:
        raise RuntimeError(
            "source dataset top-level member set drifted: "
            f"expected={sorted(expected_names)!r} "
            f"observed={sorted(observed_names)!r}"
        )

    for name in EXPECTED_CONTROL_FILES:
        path = dataset_root / name
        expected_sha256 = EXPECTED_CONTROL_SHA256[name]
        if file_sha256(path) != expected_sha256:
            raise RuntimeError(
                f"source dataset control identity drifted: {name}"
            )

    source_receipt = json.loads(
        (dataset_root / "source_packaging_receipt.json").read_text(
            encoding="utf-8"
        )
    )
    if source_receipt != EXPECTED_SOURCE_RECEIPT:
        raise RuntimeError("source packaging receipt contract drifted")

    sha_manifest = json.loads(
        (dataset_root / "sha256_manifest.json").read_text(
            encoding="utf-8"
        )
    )
    if sha_manifest != EXPECTED_SHA256_MANIFEST:
        raise RuntimeError("source SHA-256 manifest contract drifted")

    raw_inventory = json.loads(
        (dataset_root / "source_inventory.json").read_text(
            encoding="utf-8"
        )
    )
    if (
        not isinstance(raw_inventory, list)
        or len(raw_inventory) != EXPECTED_FILE_COUNT
    ):
        raise RuntimeError("source inventory shape or file count drifted")
    observed_inventory_paths = {
        entry.get("path")
        for entry in raw_inventory
        if isinstance(entry, dict)
    }
    if len(observed_inventory_paths) != len(raw_inventory):
        raise RuntimeError("source inventory contains duplicate paths")
    for entry in raw_inventory:
        if not isinstance(entry, dict):
            raise RuntimeError("source inventory contains an invalid entry")
        normalized_relative_path(str(entry.get("path")))
        if (
            not isinstance(entry.get("size_bytes"), int)
            or entry["size_bytes"] < 0
        ):
            raise RuntimeError("source inventory contains an invalid size")
        if (
            not isinstance(entry.get("sha256"), str)
            or len(entry["sha256"]) != 64
        ):
            raise RuntimeError("source inventory contains an invalid SHA-256")
        if (
            not isinstance(entry.get("git_blob_sha"), str)
            or len(entry["git_blob_sha"]) != 40
        ):
            raise RuntimeError("source inventory contains an invalid Git blob id")
        if not isinstance(entry.get("executable"), bool):
            raise RuntimeError(
                "source inventory contains an invalid executable flag"
            )
    return raw_inventory


def directory_identity(entries: list[dict[str, object]]) -> str:
    identity_entries = [
        {
            "path": entry["path"],
            "sha256": entry["sha256"],
            "size_bytes": entry["size_bytes"],
        }
        for entry in entries
    ]
    return hashlib.sha256(
        canonical_json(
            {"schema_version": "1.0.0", "files": identity_entries}
        ).encode("utf-8")
    ).hexdigest()


def inspect_expanded_source(
    expanded_root: Path,
) -> list[dict[str, object]]:
    entries: list[dict[str, object]] = []
    total_bytes = 0
    for path in sorted(
        expanded_root.rglob("*"),
        key=lambda item: item.as_posix(),
    ):
        if path.is_symlink():
            raise RuntimeError("expanded source contains a symbolic link")
        metadata = path.stat()
        if path.is_dir():
            continue
        if not stat.S_ISREG(metadata.st_mode):
            raise RuntimeError(
                "expanded source contains a non-regular member"
            )
        relative_path = path.relative_to(expanded_root).as_posix()
        normalized_relative_path(relative_path)
        if relative_path.lower().endswith(ARCHIVE_SUFFIXES):
            raise RuntimeError("expanded source contains a nested archive")
        total_bytes += metadata.st_size
        if total_bytes > MAXIMUM_TOTAL_BYTES:
            raise RuntimeError("expanded source exceeds the byte budget")
        entries.append(
            {
                "path": relative_path,
                "sha256": file_sha256(path),
                "size_bytes": metadata.st_size,
            }
        )
        if len(entries) > MAXIMUM_FILES:
            raise RuntimeError("expanded source exceeds the file-count budget")
    if total_bytes != EXPECTED_TOTAL_BYTES:
        raise RuntimeError("expanded source total bytes drifted")
    return entries


def reconstruct_exact_archive(
    expanded_root: Path,
    inventory: list[dict[str, object]],
) -> Path:
    if RECOVERED_ARCHIVE_PATH.exists():
        raise RuntimeError("recovered archive staging path already exists")
    observed_entries = inspect_expanded_source(expanded_root)
    expected_entries = [
        {
            "path": entry["path"],
            "sha256": entry["sha256"],
            "size_bytes": entry["size_bytes"],
        }
        for entry in inventory
    ]
    if observed_entries != expected_entries:
        raise RuntimeError("expanded source inventory drifted")

    with zipfile.ZipFile(
        RECOVERED_ARCHIVE_PATH,
        mode="x",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=9,
        strict_timestamps=True,
    ) as archive:
        for entry in inventory:
            relative_path = normalized_relative_path(str(entry["path"]))
            source_path = expanded_root.joinpath(*relative_path.parts)
            if not source_path.is_file() or source_path.is_symlink():
                raise RuntimeError(
                    "expanded source member is missing or non-regular"
                )
            payload = source_path.read_bytes()
            if (
                len(payload) != entry["size_bytes"]
                or hashlib.sha256(payload).hexdigest() != entry["sha256"]
            ):
                raise RuntimeError(
                    "expanded source member identity drifted"
                )
            info = zipfile.ZipInfo(
                relative_path.as_posix(),
                date_time=ZIP_TIMESTAMP,
            )
            info.create_system = 3
            info.compress_type = zipfile.ZIP_DEFLATED
            mode = 0o100755 if entry["executable"] else 0o100644
            info.external_attr = mode << 16
            archive.writestr(
                info,
                payload,
                compress_type=zipfile.ZIP_DEFLATED,
                compresslevel=9,
            )

    if (
        file_sha256(RECOVERED_ARCHIVE_PATH)
        != EXPECTED_CONTROL_SHA256[EXPECTED_ARCHIVE_NAME]
    ):
        raise RuntimeError(
            "reconstructed source archive identity drifted"
        )
    return RECOVERED_ARCHIVE_PATH


def validate_archive(
    archive_path: Path,
    inventory: list[dict[str, object]],
) -> tuple[zipfile.ZipInfo, ...]:
    if (
        file_sha256(archive_path)
        != EXPECTED_CONTROL_SHA256[EXPECTED_ARCHIVE_NAME]
    ):
        raise RuntimeError("source archive identity drifted")
    expected_by_path = {
        str(entry["path"]): entry for entry in inventory
    }
    with zipfile.ZipFile(archive_path) as archive:
        members = tuple(archive.infolist())
        names = tuple(member.filename for member in members)
        if len(names) != len(set(names)):
            raise RuntimeError("source archive contains duplicate members")
        if names != tuple(sorted(expected_by_path)):
            raise RuntimeError(
                "source archive member order or path set drifted"
            )
        for member in members:
            member_path = normalized_relative_path(member.filename)
            if member.flag_bits & 0x1:
                raise RuntimeError(
                    "source archive contains an encrypted member"
                )
            if member.is_dir():
                raise RuntimeError(
                    "source archive contains an unexpected directory member"
                )
            unix_mode = member.external_attr >> 16
            if stat.S_IFMT(unix_mode) != stat.S_IFREG:
                raise RuntimeError(
                    "source archive contains a non-regular member"
                )
            expected = expected_by_path[member_path.as_posix()]
            expected_mode = (
                0o100755 if expected["executable"] else 0o100644
            )
            if unix_mode != expected_mode:
                raise RuntimeError("source archive member mode drifted")
            if member.date_time != ZIP_TIMESTAMP:
                raise RuntimeError(
                    "source archive member timestamp drifted"
                )
            if member.compress_type != zipfile.ZIP_DEFLATED:
                raise RuntimeError(
                    "source archive compression method drifted"
                )
            if member.filename.lower().endswith(ARCHIVE_SUFFIXES):
                raise RuntimeError(
                    "source archive contains a nested archive"
                )
            if member.file_size != expected["size_bytes"]:
                raise RuntimeError("source archive member size drifted")
            payload = archive.read(member)
            if (
                hashlib.sha256(payload).hexdigest()
                != expected["sha256"]
            ):
                raise RuntimeError(
                    "source archive member identity drifted"
                )
        return members


def extract_archive(
    archive_path: Path,
    inventory: list[dict[str, object]],
) -> None:
    expected_by_path = {
        str(entry["path"]): entry for entry in inventory
    }
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            relative_path = normalized_relative_path(member.filename)
            destination = STAGING_HARNESS_ROOT.joinpath(
                *relative_path.parts
            )
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member, "r") as source:
                with destination.open("xb") as target:
                    shutil.copyfileobj(
                        source,
                        target,
                        length=1024 * 1024,
                    )
            expected = expected_by_path[relative_path.as_posix()]
            destination.chmod(
                0o755 if expected["executable"] else 0o644
            )


def inspect_materialized_tree() -> tuple[list[dict[str, object]], int]:
    entries: list[dict[str, object]] = []
    total_bytes = 0
    for path in sorted(
        STAGING_HARNESS_ROOT.rglob("*"),
        key=lambda item: item.as_posix(),
    ):
        if path.is_symlink():
            raise RuntimeError(
                "materialized harness contains a symbolic link"
            )
        metadata = path.stat()
        if path.is_dir():
            continue
        if not stat.S_ISREG(metadata.st_mode):
            raise RuntimeError(
                "materialized harness contains a non-regular member"
            )
        relative_path = path.relative_to(
            STAGING_HARNESS_ROOT
        ).as_posix()
        if relative_path.lower().endswith(ARCHIVE_SUFFIXES):
            raise RuntimeError(
                "materialized harness contains a nested archive"
            )
        total_bytes += metadata.st_size
        if total_bytes > MAXIMUM_TOTAL_BYTES:
            raise RuntimeError(
                "materialized harness exceeds the byte budget"
            )
        entries.append(
            {
                "path": relative_path,
                "sha256": file_sha256(path),
                "size_bytes": metadata.st_size,
            }
        )
        if len(entries) > MAXIMUM_FILES:
            raise RuntimeError(
                "materialized harness exceeds the file-count budget"
            )
    return entries, total_bytes


def materialize() -> dict[str, object]:
    if (
        FINAL_BUNDLE_ROOT.exists()
        or STAGING_BUNDLE_ROOT.exists()
        or RECOVERED_ARCHIVE_PATH.exists()
    ):
        raise RuntimeError(
            "materializer output or staging state already exists"
        )

    dataset_root, input_mode, representation_path = (
        resolve_source_package()
    )
    inventory = validate_dataset_controls(dataset_root, input_mode)
    completed = False
    try:
        if input_mode == "exact_archive_with_control_files":
            archive_path = representation_path
        else:
            archive_path = reconstruct_exact_archive(
                representation_path,
                inventory,
            )

        validate_archive(archive_path, inventory)
        STAGING_HARNESS_ROOT.mkdir(parents=True)
        extract_archive(archive_path, inventory)
        observed_entries, observed_total_bytes = (
            inspect_materialized_tree()
        )
        expected_entries = [
            {
                "path": entry["path"],
                "sha256": entry["sha256"],
                "size_bytes": entry["size_bytes"],
            }
            for entry in inventory
        ]
        if observed_entries != expected_entries:
            raise RuntimeError(
                "materialized harness inventory drifted"
            )
        if len(observed_entries) != EXPECTED_FILE_COUNT:
            raise RuntimeError(
                "materialized harness file count drifted"
            )
        if observed_total_bytes != EXPECTED_TOTAL_BYTES:
            raise RuntimeError(
                "materialized harness total bytes drifted"
            )
        observed_directory_sha256 = directory_identity(
            observed_entries
        )
        if (
            observed_directory_sha256
            != EXPECTED_DIRECTORY_SHA256
        ):
            raise RuntimeError(
                "materialized harness directory identity drifted"
            )
        receipt = {
            "schema_version": "1.0.0",
            "status": "CURRENT_CU129_HARNESS_MATERIALIZED",
            "producer_notebook_name": NOTEBOOK_NAME,
            "producer_output_directory": PRODUCER_OUTPUT_DIRECTORY,
            "source_commit": EXPECTED_SOURCE_COMMIT,
            "input_dataset_name": DATASET_NAME,
            "input_mode": input_mode,
            "archive_filename": EXPECTED_ARCHIVE_NAME,
            "archive_sha256": EXPECTED_CONTROL_SHA256[
                EXPECTED_ARCHIVE_NAME
            ],
            "source_inventory_sha256": EXPECTED_CONTROL_SHA256[
                "source_inventory.json"
            ],
            "source_receipt_sha256": EXPECTED_CONTROL_SHA256[
                "source_packaging_receipt.json"
            ],
            "source_sha256_manifest_sha256": EXPECTED_CONTROL_SHA256[
                "sha256_manifest.json"
            ],
            "output_directory": EXPECTED_OUTPUT_DIRECTORY,
            "directory_sha256": observed_directory_sha256,
            "file_count": len(observed_entries),
            "total_bytes": observed_total_bytes,
            "nested_archives_present": False,
            "symlinks_present": False,
            "network_access_performed": False,
            "package_installation_performed": False,
            "gpu_execution_performed": False,
            "model_loaded": False,
            "worker_started": False,
            "model_requests_performed": 0,
            "benchmark_trajectory_requests_performed": 0,
            "authorization_issued": False,
        }
        STAGING_RECEIPT_PATH.write_text(
            canonical_json(receipt),
            encoding="utf-8",
        )
        STAGING_BUNDLE_ROOT.replace(FINAL_BUNDLE_ROOT)
        completed = True
        return receipt
    finally:
        RECOVERED_ARCHIVE_PATH.unlink(missing_ok=True)
        if not completed and STAGING_BUNDLE_ROOT.exists():
            shutil.rmtree(STAGING_BUNDLE_ROOT)


if len(NOTEBOOK_NAME) > 50:
    raise RuntimeError("Kaggle notebook name exceeds 50 characters")
result = materialize()
print("status=CURRENT_CU129_HARNESS_MATERIALIZED")
print(f"producer_output_directory={PRODUCER_OUTPUT_DIRECTORY}")
print(f"input_mode={result['input_mode']}")
print(f"output_directory={EXPECTED_OUTPUT_DIRECTORY}")
print(f"file_count={result['file_count']}")
print(f"total_bytes={result['total_bytes']}")
print(f"directory_sha256={result['directory_sha256']}")
print("gpu_execution_performed=false")
print("package_installation_performed=false")
print("model_requests_performed=0")
print("authorization_issued=false")
print("save_this_notebook_output=true")
